![image_1780142384809.png](./image_1780142384809.png "image_1780142384809.png")

![image_1780142403019.png](./image_1780142403019.png "image_1780142403019.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
# Initialize Spark
spark = SparkSession.builder.appName("OrdersData").getOrCreate()

# Create DataFrame
orders_data = [
    (1, 1001, "Electronics"),
    (2, 1001, "Clothing"),
    (3, 1001, "Food"),
    (4, 1001, "Books"),
    (5, 1002, "Electronics"),
    (6, 1002, "Food"),
    (7, 1003, "Books"),
    (8, 1003, "Electronics"),
    (9, 1003, "Clothing"),
    (10, 1003, "Food"),
    (11, 1004, "Electronics"),
    (12, 1004, "Electronics"),
    (13, 1004, "Clothing"),
]

orders_df = spark.createDataFrame(orders_data, ["id", "customer_id", "category"])

# Show DataFrame
orders_df.show()


In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as f

result_df = (
    orders_df.groupBy("customer_id").agg(
        f.countDistinct("category").alias("distinct_buy")
    )
    .crossJoin(
        orders_df.select(f.countDistinct("category").alias("category_dist_count"))
    )
    .filter(f.col("distinct_buy")==f.col("category_dist_count"))
    .select("customer_id")
)
result_df.show()


In [0]:
category_dist_count = orders_df.select(
    f.countDistinct("category").alias("category_dist_count")
).collect()[0]["category_dist_count"]

result_df = (
    orders_df.groupBy("customer_id")
    .agg(f.countDistinct("category").alias("distinct_buy"))
    .filter(f.col("distinct_buy") == category_dist_count)
    .select("customer_id")
    .orderBy("customer_id")
)
result_df.show()